# AVRIL — Fine-tune YOLOv8-nano for Desktop UI Detection

Train a lightweight UI element detector on Google Colab (free T4 GPU).

**Workflow:**
1. Upload `ui_dataset.zip` to Google Drive (created by `split_dataset.py --zip`)
2. Run all cells below
3. Download `ui_detect.pt` back to your machine → `ai-model/models/ui_detect.pt`
4. Restart Flask server — `ui_parser.py` auto-loads the model

**Classes (16):** button, input, icon, menu, card, list, checkbox, radio, dropdown, toggle, link, image, text, header, nav, search_bar

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install ultralytics (includes YOLOv8)
!pip install -q ultralytics

## 2. Mount Google Drive & Extract Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile

# === EDIT THIS PATH if your zip is in a different Drive folder ===
ZIP_PATH = "/content/drive/MyDrive/ui_dataset.zip"

DATASET_DIR = "/content/ui_detect"

if not os.path.isfile(ZIP_PATH):
    print(f"ERROR: {ZIP_PATH} not found!")
    print("Upload ui_dataset.zip to your Google Drive root first.")
    print("Create it locally with: python scripts/split_dataset.py --zip")
else:
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATASET_DIR)
    # Show contents
    for root, dirs, files in os.walk(DATASET_DIR):
        level = root.replace(DATASET_DIR, '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 2:
            for f in files[:5]:
                print(f"{indent}  {f}")
            if len(files) > 5:
                print(f"{indent}  ... and {len(files)-5} more")

In [ ]:
# Count images
import glob
n_train = len(glob.glob(f"{DATASET_DIR}/train/images/*.png"))
n_val = len(glob.glob(f"{DATASET_DIR}/val/images/*.png"))
print(f"Train: {n_train} images")
print(f"Val:   {n_val} images")
print(f"Total: {n_train + n_val}")

## 3. Patch data.yaml paths for Colab

In [ ]:
# Rewrite data.yaml with absolute Colab paths
DATA_YAML = f"{DATASET_DIR}/data.yaml"

yaml_content = f"""# Auto-patched for Google Colab
path: {DATASET_DIR}
train: train/images
val: val/images

names:
  0: button
  1: input
  2: icon
  3: menu
  4: card
  5: list
  6: checkbox
  7: radio
  8: dropdown
  9: toggle
  10: link
  11: image
  12: text
  13: header
  14: nav
  15: search_bar
"""

with open(DATA_YAML, 'w') as f:
    f.write(yaml_content)

print(f"Wrote {DATA_YAML}")
print(yaml_content)

## 4. Train YOLOv8-nano

In [ ]:
from ultralytics import YOLO

# Load YOLOv8-nano with COCO pretrained weights
model = YOLO("yolov8n.pt")

# Train with UI-specific augmentation
results = model.train(
    data=DATA_YAML,
    epochs=100,
    batch=16,          # Colab T4 has 16GB — can handle batch=16
    imgsz=640,
    patience=15,       # early stopping
    device=0,          # GPU

    project="/content/runs",
    name="ui_detect",
    exist_ok=True,

    # UI-specific: no rotation, no flips, reduced mosaic
    degrees=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.3,
    hsv_h=0.005,
    hsv_s=0.3,
    hsv_v=0.3,
    scale=0.3,
    translate=0.1,

    workers=2,
    verbose=True,
    save=True,
    plots=True,
)

## 5. Evaluate

In [ ]:
# Validate on val set
metrics = model.val()
print(f"\nmAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# Show training curves
from IPython.display import Image, display
results_img = "/content/runs/ui_detect/results.png"
if os.path.isfile(results_img):
    display(Image(filename=results_img, width=900))
else:
    print("results.png not found")

In [ ]:
# Show confusion matrix
cm_img = "/content/runs/ui_detect/confusion_matrix.png"
if os.path.isfile(cm_img):
    display(Image(filename=cm_img, width=700))
else:
    print("confusion_matrix.png not found")

## 6. Export & Download

In [ ]:
import shutil

BEST_PT = "/content/runs/ui_detect/weights/best.pt"
EXPORT_NAME = "ui_detect.pt"

# Copy to Drive for safekeeping
drive_dest = f"/content/drive/MyDrive/{EXPORT_NAME}"
shutil.copy2(BEST_PT, drive_dest)
size_mb = os.path.getsize(drive_dest) / (1024 * 1024)
print(f"Saved to Google Drive: {drive_dest} ({size_mb:.1f} MB)")

# Also trigger browser download
from google.colab import files
files.download(BEST_PT)

## 7. Done!

Copy `ui_detect.pt` to your local machine:

```bash
# Place the downloaded file at:
cp ~/Downloads/ui_detect.pt ~/divyansh/ai-model/models/ui_detect.pt

# Restart the Flask server
cd ~/divyansh/ai-model && python main.py

# Test — should show source: yolo
curl http://localhost:8000/ui-parse
```

The model is loaded automatically by `ui_parser.py` on first use.